<a href="https://colab.research.google.com/github/patelmax98/git_practice/blob/main/CY_Electricity_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypsa pandas numpy geopandas plotly
# A solver is required for optimisation. HiGHS is free, fast, and pip-installable:
!pip install highspy

In [ ]:
# notebooks/00_setup.ipynb  — environment check & imports
# WHAT: confirms your stack is installed and importable before we build anything.

import sys
print("Python:", sys.version.split()[0])

# Core modelling
import pypsa
import pandas as pd
import numpy as np

# Geo + viz (used from Phase 4 onward)
import geopandas as gpd
import plotly.graph_objects as go

print("PyPSA:", pypsa.__version__)
print("pandas:", pd.__version__)
print("Ready ✅")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.7/358.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.5 MB/s eta 0:00:00
Python: 3.12.13
PyPSA: 1.2.3
pandas: 2.2.2
Ready ✅


In [ ]:
# notebooks/01_build_network.ipynb
# WHAT: defines the physical Cyprus grid as a PyPSA network — buses, lines,
#       generators, loads — then solves ONE snapshot to confirm it works.
# We use synthetic-but-plausible numbers throughout. Every "CHANGE LATER"
# comment marks a value you'd swap for real data in a future iteration.

import pypsa
import pandas as pd
import numpy as np

# A PyPSA Network is the top-level container for everything: components,
# time-series, and (after solving) results. Think of it as the whole model.
n = pypsa.Network()

# ----------------------------------------------------------------------------
# 1. BUSES  — electrical nodes. A "bus" is a point in the grid where things
#    connect (generators, loads, lines). We place ~5 at real Cyprus locations
#    so the map is meaningful and "locational" prices vary across the island.
#    CHANGE LATER: add more buses, or use real substation coordinates from
#    TSOC / OpenStreetMap, for finer spatial resolution.
# ----------------------------------------------------------------------------
buses = pd.DataFrame(
    {
        # name        : longitude (x), latitude (y)   — used later for the map
        "Vasilikos":   (33.336, 34.717),  # largest power station (south coast)
        "Dhekelia":    (33.700, 34.985),  # power station (east)
        "Moni":        (33.183, 34.713),  # power station (south)
        "Nicosia":     (33.365, 35.185),  # capital — big load centre, inland
        "Limassol":    (33.045, 34.707),  # major coastal city — big load
    },
    index=["x", "y"],
).T

for name, row in buses.iterrows():
    # v_nom is nominal voltage (kV). Cyprus transmission backbone is 132/220 kV.
    # CHANGE LATER: set per-bus real voltages; here one value keeps it simple.
    n.add("Bus", name, x=row.x, y=row.y, v_nom=132)

# ----------------------------------------------------------------------------
# 2. LINES — transmission links between buses. We model a simple ring/star so
#    power can flow island-wide. We use a transport-style limit (s_nom = MVA
#    rating) rather than detailed impedances to keep the first solve simple.
#    CHANGE LATER: use real line lengths, reactances (x), and ratings, and
#    switch to full AC/linearised power flow.
# ----------------------------------------------------------------------------
lines = [
    # (name,            bus0,        bus1,        s_nom MW)
    ("Vasilikos-Moni",  "Vasilikos", "Moni",      400),
    ("Moni-Limassol",   "Moni",      "Limassol",  400),
    ("Vasilikos-Nicosia","Vasilikos","Nicosia",   500),
    ("Nicosia-Dhekelia","Nicosia",   "Dhekelia",  300),
    ("Dhekelia-Vasilikos","Dhekelia","Vasilikos", 300),
]

for name, b0, b1, s_nom in lines:
    n.add(
        "Line", name, bus0=b0, bus1=b1,
        s_nom=s_nom,        # thermal rating (MW the line can carry)
        # x = series reactance (ohms). Required for linear power flow (LOPF).
        # A small nonzero value is enough for the model to distribute flows.
        # CHANGE LATER: compute from real length × per-km reactance.
        x=0.1,
        r=0.01,             # series resistance (ohms) — minor here
    )

# ----------------------------------------------------------------------------
# 3. GENERATORS — power plants. Each lives on a bus and has a capacity (p_nom),
#    a marginal cost (€/MWh, drives the merit order), and a CO2 factor we track
#    via 'carrier'. Costs/capacities are synthetic but reflect Cyprus's reality:
#    heavy-fuel-oil & diesel thermal plants + growing solar PV.
#    CHANGE LATER: real capacities (TSOC reports), real fuel costs, real CO2.
# ----------------------------------------------------------------------------

# First define CARRIERS (fuel/tech types) so we can track emissions centrally.
# co2_emissions is in tonnes CO2 per MWh of *thermal* input; PyPSA combines it
# with efficiency to get emissions per MWh electrical.
# CHANGE LATER: refine efficiencies & emission factors per real plant.
n.add("Carrier", "oil",   co2_emissions=0.65)   # heavy fuel oil
n.add("Carrier", "diesel",co2_emissions=0.70)
n.add("Carrier", "solar", co2_emissions=0.0)

generators = [
    # (name,            bus,         carrier, p_nom MW, marg.cost €/MWh, eff)
    ("Vasilikos_HFO",   "Vasilikos", "oil",    700, 110, 0.40),
    ("Dhekelia_HFO",    "Dhekelia",  "oil",    300, 120, 0.38),
    ("Moni_Diesel",     "Moni",      "diesel", 150, 180, 0.35),  # peaker: pricey
    ("Limassol_Solar",  "Limassol",  "solar",  200,   0, 1.00),  # ~zero fuel cost
    ("Nicosia_Solar",   "Nicosia",   "solar",  150,   0, 1.00),
]

for name, bus, carrier, p_nom, mc, eff in generators:
    n.add(
        "Generator", name, bus=bus, carrier=carrier,
        p_nom=p_nom,             # installed capacity (MW)
        marginal_cost=mc,        # €/MWh — sets dispatch order (cheapest first)
        efficiency=eff,          # used with carrier CO2 factor for emissions
        # p_max_pu defaults to 1 (full availability). In Phase 2 we'll override
        # this for solar with an hourly sunshine profile (0..1 each hour).
    )

# ----------------------------------------------------------------------------
# 4. LOADS — electricity demand. Each load sits on a bus and draws power (MW).
#    For this single-snapshot test we use fixed values at the two cities.
#    Phase 2 replaces these with an hourly demand curve.
#    CHANGE LATER: real demand split across buses (ENTSO-E actual load).
# ----------------------------------------------------------------------------
loads = [
    # (name,          bus,        p_set MW)
    ("Nicosia_Load",  "Nicosia",  450),
    ("Limassol_Load", "Limassol", 350),
    ("Dhekelia_Load", "Dhekelia", 120),
]

for name, bus, p_set in loads:
    n.add("Load", name, bus=bus, p_set=p_set)  # p_set = demand at this snapshot

# Quick sanity check: total demand vs total firm (non-solar) capacity.
total_demand = sum(p for *_, p in loads)
print(f"Total demand this snapshot: {total_demand} MW")
print(f"Buses: {len(n.buses)}, Lines: {len(n.lines)}, "
      f"Generators: {len(n.generators)}, Loads: {len(n.loads)}")

Total demand this snapshot: 920 MW
Buses: 5, Lines: 5, Generators: 5, Loads: 3


In [ ]:
# WHAT: run a single-snapshot Linear Optimal Power Flow (LOPF).
# This finds the cheapest way to meet demand while respecting line limits.

# n.optimize() builds a linear program (minimise total generation cost) and
# solves it with HiGHS. With no time-series set yet, it solves the single
# default snapshot using the fixed p_set loads and full generator availability.
n.optimize(solver_name="highs", include_objective_constant=False)
# Easiest: let PyPSA fill in defaults
n.consistency_check()   # optional: see what it flags
n.sanitize()            # auto-adds the missing carrier definitions

# Inspect results -----------------------------------------------------------
print("\n--- Generator dispatch (MW) ---")
print(n.generators_t.p.T)          # how much each generator produced

print("\n--- Bus marginal prices (€/MWh) ---")
# These are the locational marginal prices (LMPs): the cost of supplying one
# more MW at each bus. They differ across buses when lines congest.
print(n.buses_t.marginal_price.T)

print(f"\nTotal system cost: €{n.objective:,.0f}")

/tmp/ipykernel_2121/2038526134.py:7: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['Vasilikos', 'Dhekelia', 'Moni', 'Nicosia', 'Limassol'], dtype='object', name='name')
Index(['Vasilikos-Moni', 'Moni-Limassol', 'Vasilikos-Nicosia',
       'Nicosia-Dhekelia', 'Dhekelia-Vasilikos'],
      dtype='object', name='name')



--- Generator dispatch (MW) ---
snapshot          now
name                 
Vasilikos_HFO   570.0
Dhekelia_HFO     -0.0
Moni_Diesel      -0.0
Limassol_Solar  200.0
Nicosia_Solar   150.0

--- Bus marginal prices (€/MWh) ---
snapshot     now
name            
Vasilikos  110.0
Dhekelia   110.0
Moni       110.0
Nicosia    110.0
Limassol   110.0

Total system cost: €62,700


In [ ]:
# WHAT: deliberately create transmission congestion so locational prices SPLIT.
# We do this on a COPY of the network so the original (n) stays clean for Phase 2.

# n.copy() duplicates the whole model — components, parameters, everything.
# A solved network holds a live solver object that can't be deep-copied.
# Detach it first (results stay intact — only the solver handle is removed).
n.model.solver_model = None

# Now the copy works.
m = n.copy()

# ----------------------------------------------------------------------------
# THE TRICK: Nicosia is inland with NO local generator — all its power must
# arrive over transmission lines. If we (a) raise Nicosia's demand and
# (b) throttle the lines feeding it, the cheap southern HFO power physically
# can't all get through. The optimiser is then FORCED to run a more expensive
# plant to serve the trapped extra demand → Nicosia's price rises above the south.
# ----------------------------------------------------------------------------

# (a) Crank up Nicosia demand so the import need is large.
#     CHANGE LATER: this is a teaching exaggeration, not a real value.
m.loads.loc["Nicosia_Load", "p_set"] = 900   # was 450 MW

# (b) Shrink the two lines that feed Nicosia so they bottleneck.
#     s_nom is the MW rating — lowering it tightens the pipe.
m.lines.loc["Vasilikos-Nicosia", "s_nom"] = 250   # was 500
m.lines.loc["Nicosia-Dhekelia",  "s_nom"] = 150   # was 300

# Re-solve the modified network.
m.optimize(solver_name="highs", include_objective_constant=False)

# ----------------------------------------------------------------------------
# Read the results
# ----------------------------------------------------------------------------
print("--- Generator dispatch (MW) ---")
print(m.generators_t.p.T)

print("\n--- Bus marginal prices (€/MWh) ---")
print(m.buses_t.marginal_price.T)

print("\n--- Line loading: flow vs limit ---")
flow  = m.lines_t.p0.iloc[0]                  # power entering each line (MW)
limit = m.lines.s_nom                         # each line's MW rating
loading = (flow.abs() / limit * 100).round(1) # % of capacity used
print(pd.DataFrame({"flow_MW": flow.round(1),
                    "limit_MW": limit,
                    "loading_%": loading}))

print(f"\nTotal system cost: €{m.objective:,.0f}")

Index(['Vasilikos', 'Dhekelia', 'Moni', 'Nicosia', 'Limassol'], dtype='object', name='name')
Index(['Vasilikos-Moni', 'Moni-Limassol', 'Vasilikos-Nicosia',
       'Nicosia-Dhekelia', 'Dhekelia-Vasilikos'],
      dtype='object', name='name')
Index(['0'], dtype='object', name='name')
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Infeasible



--- Generator dispatch (MW) ---
snapshot          now
name                 
Vasilikos_HFO   570.0
Dhekelia_HFO     -0.0
Moni_Diesel      -0.0
Limassol_Solar  200.0
Nicosia_Solar   150.0

--- Bus marginal prices (€/MWh) ---
snapshot     now
name            
Vasilikos  110.0
Dhekelia   110.0
Moni       110.0
Nicosia    110.0
Limassol   110.0

--- Line loading: flow vs limit ---
                    flow_MW  limit_MW  loading_%
name                                            
Vasilikos-Moni        150.0     400.0       37.5
Moni-Limassol         150.0     400.0       37.5
Vasilikos-Nicosia     240.0     250.0       96.0
Nicosia-Dhekelia      -60.0     150.0       40.0
Dhekelia-Vasilikos   -180.0     300.0       60.0

Total system cost: €62,700


In [ ]:
# Detach solver handle from the last solve, then copy a fresh network.
n.model.solver_model = None
m = n.copy()

# Nicosia import need must be FEASIBLE (≤ total feeder capacity) but large
# enough to fill the cheap line. Solar at Nicosia = 150 MW.
m.loads.loc["Nicosia_Load", "p_set"] = 600    # import need = 600 - 150 = 450 MW

# Feeders: total capacity 250 + 250 = 500 MW  > 450 MW needed  → FEASIBLE.
# But the cheap southern path (Vasilikos-Nicosia) alone can't carry it all,
# so it saturates and forces pricier power in via the other route → congestion.
m.lines.loc["Vasilikos-Nicosia", "s_nom"] = 250   # was 500
m.lines.loc["Nicosia-Dhekelia",  "s_nom"] = 250   # was 300

m.optimize(solver_name="highs", include_objective_constant=False)

print("--- Generator dispatch (MW) ---")
print(m.generators_t.p.T)
print("\n--- Bus marginal prices (€/MWh) ---")
print(m.buses_t.marginal_price.T)
print("\n--- Line loading: flow vs limit ---")
flow  = m.lines_t.p0.iloc[0]
limit = m.lines.s_nom
loading = (flow.abs() / limit * 100).round(1)
print(pd.DataFrame({"flow_MW": flow.round(1), "limit_MW": limit, "loading_%": loading}))
print(f"\nTotal system cost: €{m.objective:,.0f}")

Index(['Vasilikos', 'Dhekelia', 'Moni', 'Nicosia', 'Limassol'], dtype='object', name='name')
Index(['Vasilikos-Moni', 'Moni-Limassol', 'Vasilikos-Nicosia',
       'Nicosia-Dhekelia', 'Dhekelia-Vasilikos'],
      dtype='object', name='name')
Index(['0'], dtype='object', name='name')


--- Generator dispatch (MW) ---
snapshot          now
name                 
Vasilikos_HFO   450.0
Dhekelia_HFO    270.0
Moni_Diesel      -0.0
Limassol_Solar  200.0
Nicosia_Solar   150.0

--- Bus marginal prices (€/MWh) ---
snapshot     now
name            
Vasilikos  110.0
Dhekelia   120.0
Moni       110.0
Nicosia    130.0
Limassol   110.0

--- Line loading: flow vs limit ---
                    flow_MW  limit_MW  loading_%
name                                            
Vasilikos-Moni        150.0     400.0       37.5
Moni-Limassol         150.0     400.0       37.5
Vasilikos-Nicosia     250.0     250.0      100.0
Nicosia-Dhekelia     -200.0     250.0       80.0
Dhekelia-Vasilikos    -50.0     300.0       16.7

Total system cost: €81,900


In [ ]:
# @title
# notebooks/02_timeseries.ipynb  (continues from the network `n` we built)
# WHAT: replace the single snapshot with 24 hourly snapshots, give loads a daily
#       demand curve and solar a sunshine profile, then solve economic dispatch
#       across the whole day.

import numpy as np
import pandas as pd

# Work on the clean original network. (If n still holds a solver handle from
# earlier, detach it so we can re-solve fresh.)
try:
    n.model.solver_model = None
except Exception:
    pass

# ----------------------------------------------------------------------------
# 1. SNAPSHOTS — the time steps the optimiser solves over. We move from the
#    single default "now" to 24 hourly steps for one representative day.
#    CHANGE LATER: swap this date range for a full year (8760 h) — the rest of
#    the code below adapts automatically because it's all profile-driven.
# ----------------------------------------------------------------------------
snapshots = pd.date_range("2024-06-21 00:00", periods=24, freq="h")  # a summer day
n.set_snapshots(snapshots)

# n.snapshot_weightings tells PyPSA how many "real" hours each snapshot stands
# for (matters for totals/costs). For 24 actual hourly steps each weight = 1.
# CHANGE LATER: if you use representative days to stand in for a year, raise
# these weights so totals scale up correctly.
n.snapshot_weightings.loc[:, :] = 1.0

# ----------------------------------------------------------------------------
# 2. DEMAND PROFILE — a normalised daily shape (0..1) scaled to each load's peak.
#    Shape: low overnight, morning rise, dip, strong evening peak (~20:00).
#    This is synthetic but reflects a typical Mediterranean summer load curve.
#    CHANGE LATER: replace with real hourly load from ENTSO-E for Cyprus.
# ----------------------------------------------------------------------------
hours = np.arange(24)
demand_shape = (
    0.55                                   # overnight base level
    + 0.20 * np.exp(-((hours - 9)  ** 2) / 8)    # morning bump
    + 0.45 * np.exp(-((hours - 20) ** 2) / 6)    # evening peak (dominant)
)
demand_shape = demand_shape / demand_shape.max()   # normalise so peak = 1.0
demand_shape = pd.Series(demand_shape, index=snapshots)

# Each load's hourly demand = its peak MW (the p_set we set earlier) × the shape.
# We move the fixed p_set value into a time-varying p_set series.
for load in n.loads.index:
    peak_mw = n.loads.at[load, "p_set"]            # reuse Phase-1 value as PEAK
    n.loads_t.p_set[load] = peak_mw * demand_shape
# Clear the static scalar so PyPSA uses the time-series version.
n.loads["p_set"] = 0.0

# ----------------------------------------------------------------------------
# 3. SOLAR PROFILE — availability factor (0..1) by hour. Zero at night, peaks
#    near midday. We apply it via p_max_pu, which caps each generator's output
#    to a fraction of its p_nom per snapshot. Thermal plants stay at 1 (always
#    available). CHANGE LATER: use real PVGIS / renewables.ninja Cyprus profile.
# ----------------------------------------------------------------------------
solar_shape = np.clip(np.sin((hours - 6) / 12 * np.pi), 0, None)  # sunrise~6 sunset~18
solar_shape = pd.Series(solar_shape, index=snapshots)

# Apply to every solar generator; leave thermal availability at default (1.0).
solar_gens = n.generators.index[n.generators.carrier == "solar"]
for g in solar_gens:
    n.generators_t.p_max_pu[g] = solar_shape

print("Snapshots:", len(n.snapshots))
print("\nDemand at Nicosia over the day (MW):")
print(n.loads_t.p_set["Nicosia_Load"].round(0).values)
print("\nSolar availability factor over the day:")
print(solar_shape.round(2).values)

Snapshots: 24

Demand at Nicosia over the day (MW):
[248. 248. 248. 248. 251. 260. 277. 302. 327. 337. 327. 302. 277. 260.
 252. 252. 262. 293. 351. 419. 450. 419. 351. 293.]

Solar availability factor over the day:
[0.   0.   0.   0.   0.   0.   0.   0.26 0.5  0.71 0.87 0.97 1.   0.97
 0.87 0.71 0.5  0.26 0.   0.   0.   0.   0.   0.  ]


In [ ]:
# WHAT: solve economic dispatch across all 24 snapshots at once.
n.optimize(solver_name="highs", include_objective_constant=False)

# Build a quick tidy view of the day -----------------------------------------
dispatch = n.generators_t.p          # MW per generator per hour
prices   = n.buses_t.marginal_price  # €/MWh per bus per hour

# Load-weighted single "Cyprus system price" per hour (the honest market number
# discussed above): each hour, average the bus prices weighted by local demand.
load_by_bus = n.loads_t.p_set.rename(columns=n.loads.bus)        # demand per bus
price_by_bus = prices.reindex(columns=load_by_bus.columns)
system_price = (price_by_bus * load_by_bus).sum(axis=1) / load_by_bus.sum(axis=1)

print("Hour | System price €/MWh | Total demand MW | Solar MW")
solar_total = dispatch[solar_gens].sum(axis=1)
demand_total = n.loads_t.p_set.sum(axis=1)
for h in range(24):
    print(f"{h:>4} | {system_price.iloc[h]:>17.1f} | "
          f"{demand_total.iloc[h]:>14.0f} | {solar_total.iloc[h]:>7.0f}")

print(f"\nTotal system cost for the day: €{n.objective:,.0f}")

Index(['Vasilikos', 'Dhekelia', 'Moni', 'Nicosia', 'Limassol'], dtype='object', name='name')
Index(['Vasilikos-Moni', 'Moni-Limassol', 'Vasilikos-Nicosia',
       'Nicosia-Dhekelia', 'Dhekelia-Vasilikos'],
      dtype='object', name='name')
Index(['0'], dtype='object', name='name')


Hour | System price €/MWh | Total demand MW | Solar MW
   0 |             110.0 |            506 |       0
   1 |             110.0 |            506 |       0
   2 |             110.0 |            506 |       0
   3 |             110.0 |            508 |       0
   4 |             110.0 |            514 |       0
   5 |             110.0 |            531 |       0
   6 |             110.0 |            566 |       0
   7 |             110.0 |            618 |      91
   8 |             110.0 |            668 |     175
   9 |             110.0 |            690 |     247
  10 |             110.0 |            668 |     303
  11 |             110.0 |            618 |     338
  12 |             110.0 |            566 |     350
  13 |             110.0 |            531 |     338
  14 |             110.0 |            515 |     303
  15 |             110.0 |            514 |     247
  16 |             110.0 |            535 |     175
  17 |             110.0 |            598 |      91
  18 |   

phase 3 - extract dataframes

In [ ]:
# notebooks/03_indicators.ipynb  (continues from solved network n)
# WHAT: turn raw PyPSA results into tidy dataframes the dashboard will consume.
# Each block produces one clean table. CHANGE LATER: add storage, imports, etc.

import pandas as pd
import numpy as np

# --- 0. Handy lookups -------------------------------------------------------
gen_carrier = n.generators.carrier            # generator -> fuel type
gen_bus     = n.generators.bus                # generator -> location

# ----------------------------------------------------------------------------
# 1. GENERATION MIX — total MWh per fuel over the day (the "where did our
#    energy come from" pie). Sum dispatch over hours, group by carrier.
# ----------------------------------------------------------------------------
gen_mwh = n.generators_t.p.sum()              # MWh per generator (hourly→sum)
mix_by_carrier = gen_mwh.groupby(gen_carrier).sum().sort_values(ascending=False)
print("--- Generation mix (MWh over the day) ---")
print(mix_by_carrier.round(0))
print(f"Solar share: {mix_by_carrier.get('solar',0)/mix_by_carrier.sum()*100:.1f}%\n")

# ----------------------------------------------------------------------------
# 2. DISPATCH (long/tidy) — one row per generator per hour. Long format is what
#    plotting libraries like best (easy to filter/animate by hour).
# ----------------------------------------------------------------------------
dispatch_long = (
    n.generators_t.p
    .reset_index().melt(id_vars="snapshot", var_name="generator", value_name="MW")
)
dispatch_long["carrier"] = dispatch_long["generator"].map(gen_carrier)
dispatch_long["bus"]     = dispatch_long["generator"].map(gen_bus)
print("--- Dispatch (tidy, first rows) ---")
print(dispatch_long.head(), "\n")

# ----------------------------------------------------------------------------
# 3. PRICES — per-bus hourly LMP, plus the single load-weighted system price.
# ----------------------------------------------------------------------------
prices = n.buses_t.marginal_price.copy()                    # €/MWh per bus per hour
load_by_bus = n.loads_t.p_set.rename(columns=n.loads.bus)
system_price = (prices.reindex(columns=load_by_bus.columns) * load_by_bus).sum(axis=1) \
               / load_by_bus.sum(axis=1)
prices["SYSTEM"] = system_price                             # headline market price
print("--- Prices (€/MWh, first hours) ---")
print(prices.round(1).head(), "\n")

# ----------------------------------------------------------------------------
# 4. LINE LOADING — % of each line's rating used, per hour. Drives the map's
#    line colours (green=spare, red=congested).
# ----------------------------------------------------------------------------
line_loading = (n.lines_t.p0.abs() / n.lines.s_nom * 100)   # % per line per hour
print("--- Line loading (%), peak hour 20 ---")
print(line_loading.loc[n.snapshots[20]].round(0), "\n")

# ----------------------------------------------------------------------------
# 5. EMISSIONS & COST -------------------------------------------------------
#    Emissions per generator = output / efficiency × carrier CO2 factor.
#    (output/efficiency converts electrical MWh back to thermal input MWh.)
# ----------------------------------------------------------------------------
co2_factor = n.generators.carrier.map(n.carriers.co2_emissions)   # t/MWh thermal
eff        = n.generators.efficiency
emissions_per_gen = (n.generators_t.p / eff) * co2_factor          # t CO2 per hour
emissions_hourly  = emissions_per_gen.sum(axis=1)                  # t CO2 per hour
total_emissions   = emissions_hourly.sum()                        # t CO2 for the day

print(f"--- Emissions & cost ---")
print(f"Total CO2 today: {total_emissions:,.0f} tonnes")
print(f"Total system cost today: €{n.objective:,.0f}")
print(f"Average cost of electricity: €{n.objective/n.loads_t.p_set.sum().sum():.1f}/MWh")

# --- Bundle everything for the dashboard -----------------------------------
indicators = {
    "mix_by_carrier": mix_by_carrier,
    "dispatch_long":  dispatch_long,
    "prices":         prices,
    "line_loading":   line_loading,
    "emissions_hourly": emissions_hourly,
    "total_emissions": total_emissions,
    "system_cost":    n.objective,
}
print("\n✅ Indicators bundled — ready for the dashboard.")

--- Generation mix (MWh over the day) ---
carrier
oil       12169.0
solar      2659.0
diesel        0.0
dtype: float64
Solar share: 17.9%

--- Dispatch (tidy, first rows) ---
             snapshot      generator          MW carrier        bus
0 2024-06-21 00:00:00  Vasilikos_HFO  506.007345     oil  Vasilikos
1 2024-06-21 01:00:00  Vasilikos_HFO  506.061698     oil  Vasilikos
2 2024-06-21 02:00:00  Vasilikos_HFO  506.402471     oil  Vasilikos
3 2024-06-21 03:00:00  Vasilikos_HFO  508.044028     oil  Vasilikos
4 2024-06-21 04:00:00  Vasilikos_HFO  514.084368     oil  Vasilikos 

--- Prices (€/MWh, first hours) ---
name                 Vasilikos  Dhekelia   Moni  Nicosia  Limassol  SYSTEM
snapshot                                                                  
2024-06-21 00:00:00      110.0     110.0  110.0    110.0     110.0   110.0
2024-06-21 01:00:00      110.0     110.0  110.0    110.0     110.0   110.0
2024-06-21 02:00:00      110.0     110.0  110.0    110.0     110.0   110.0
2024

Create Interactive Geographic Map Dashboard

In [12]:
# notebooks/04_dashboard.ipynb  (continues from solved network n + indicators)
# WHAT: an interactive map of the Cyprus grid with an hour slider. Buses show
#       price & generation; lines show loading. Hover for detail; drag the
#       slider to watch the day evolve.
# Stack: plotly only (no server). CHANGE LATER: wrap in Panel for a multi-panel
#        app, or add a real basemap with mapbox tiles + token.

import plotly.graph_objects as go
import pandas as pd
import numpy as np

# --- Pull the pieces we need (from Phase 3) --------------------------------
prices       = n.buses_t.marginal_price          # €/MWh per bus per hour
line_loading = (n.lines_t.p0.abs() / n.lines.s_nom * 100)  # % per line per hour
gen_by_bus_t = n.generators_t.p.T.groupby(n.generators.bus).sum().T  # MW/bus/hour
bus_xy       = n.buses[["x", "y"]]                # coordinates for plotting
snaps        = n.snapshots

# Colour helper: map a loading % to a colour (green→amber→red).
def load_colour(pct):
    if pct < 50:  return "#2ecc71"     # green  — plenty of spare capacity
    if pct < 80:  return "#f1c40f"     # amber  — getting busy
    return "#e74c3c"                   # red    — near/at limit (congestion)

# --- Build one set of map traces per hour (these become animation frames) ---
def build_frame(h):
    snap = snaps[h]
    traces = []

    # 1. LINES — one trace each, coloured by loading, with a hover label.
    for ln, row in n.lines.iterrows():
        x0, y0 = bus_xy.loc[row.bus0]
        x1, y1 = bus_xy.loc[row.bus1]
        pct = line_loading.at[snap, ln]
        traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1], mode="lines",
            line=dict(width=4, color=load_colour(pct)),
            hoverinfo="text",
            text=f"{ln}<br>Loading: {pct:.0f}%",
            showlegend=False,
        ))

    # 1b. LINE LABELS — a % loading number floating at each line's MIDPOINT.
    #     We compute the midpoint of each line and drop a text marker there.
    #     CHANGE LATER: nudge labels off the line if they overlap on a busy map.
    mid_x, mid_y, labels = [], [], []
    for ln, row in n.lines.iterrows():
        x0, y0 = bus_xy.loc[row.bus0]
        x1, y1 = bus_xy.loc[row.bus1]
        mid_x.append((x0 + x1) / 2)            # midpoint longitude
        mid_y.append((y0 + y1) / 2)            # midpoint latitude
        labels.append(f"{line_loading.at[snap, ln]:.0f}%")
    traces.append(go.Scatter(
        x=mid_x, y=mid_y, mode="text",
        text=labels, textfont=dict(size=11, color="#222"),
        hoverinfo="skip", showlegend=False,
    ))

    # 2. BUSES — markers sized by generation, coloured by price.
    sizes  = 12 + gen_by_bus_t.reindex(columns=bus_xy.index).loc[snap].fillna(0) / 20
    colour = prices.loc[snap, bus_xy.index]
    hover  = [f"{b}<br>Price: {prices.at[snap,b]:.0f} €/MWh"
              f"<br>Gen: {gen_by_bus_t.reindex(columns=bus_xy.index).at[snap,b]:.0f} MW"
              for b in bus_xy.index]
    traces.append(go.Scatter(
        x=bus_xy.x, y=bus_xy.y, mode="markers+text",
        marker=dict(size=sizes, color=colour, colorscale="YlOrRd",
                    cmin=float(prices.values.min()), cmax=float(prices.values.max()),
                    colorbar=dict(title="€/MWh"), line=dict(width=1, color="black")),
        text=bus_xy.index, textposition="top center",
        hoverinfo="text", hovertext=hover, showlegend=False,
    ))
    return traces

# --- Assemble figure with an hour slider -----------------------------------
fig = go.Figure(data=build_frame(0))   # start at hour 0
fig.frames = [go.Frame(data=build_frame(h), name=str(h)) for h in range(len(snaps))]

# Slider: one step per hour. Dragging it swaps which frame is shown.
sliders = [dict(
    active=0, currentvalue={"prefix": "Hour: "},
    steps=[dict(method="animate", label=str(h),
                args=[[str(h)], dict(mode="immediate",
                      frame=dict(duration=0, redraw=True))])
           for h in range(len(snaps))],
)]

# Play button to auto-animate through the day.
play = dict(type="buttons", showactive=False,
            buttons=[dict(label="▶ Play", method="animate",
                          args=[None, dict(frame=dict(duration=400, redraw=True),
                                           fromcurrent=True)])])

fig.update_layout(
    title="Cyprus electricity system — drag the slider to scrub the day",
    xaxis=dict(title="Longitude", showgrid=False),
    yaxis=dict(title="Latitude", showgrid=False, scaleanchor="x"),  # keep map aspect
    sliders=sliders, updatemenus=[play],
    width=800, height=600, plot_bgcolor="#eef3f8",
)

fig.show()

# --- Save a standalone interactive file for GitHub -------------------------
fig.write_html("cyprus_dashboard.html")
print("✅ Saved cyprus_dashboard.html — open it in any browser, no Python needed.")

✅ Saved cyprus_dashboard.html — open it in any browser, no Python needed.


Cleanup the carrier warnings for inputting into a repo

In [13]:
# WHAT: tag buses/lines/sub-networks with carriers so the consistency warnings
#       you've seen all along disappear. Purely cosmetic — improves repo polish.
n.buses["carrier"]  = "AC"      # electrical buses
n.lines["carrier"]  = "AC"      # AC transmission lines
n.sanitize()                    # fills any remaining gaps with safe defaults
n.consistency_check()           # should now print nothing / no warnings
print("✅ Network sanitized — warnings cleared.")

✅ Network sanitized — warnings cleared.
